# 18 Sequence TCN / MS-TCN Training

Purpose: train a real structured-sequence baseline on `features/structured_sequence_v1/frames.parquet`.

Required inputs:

- `manifests/bbe_events_v1.parquet`
- `features/structured_sequence_v1/manifest.parquet`
- `features/structured_sequence_v1/frames.parquet`
- optional `datasets/event_with_player_prior_v1/manifest.parquet` from notebook 13, used as a player-season mechanics prior when available

Created outputs:

- `predictions/sequence_tcn_mlb_2024_2026_v2/predictions_v1.parquet`
- `predictions/sequence_tcn_mlb_2024_2026_v2/metrics_v1.json`

This is a GPU-recommended Colab step. It keeps xBA/xwOBA missing labels masked and never trains OPS as an event head.


In [ ]:
from pathlib import Path
import os
import sys
import importlib.util


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return

    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    try:
        drive.mount(str(mountpoint))
    except ValueError as exc:
        message = str(exc)
        if 'Mountpoint must not already contain files' in message or 'already mounted' in message:
            print(f'Drive mount warning: {message}')
            if not (mountpoint / 'MyDrive').exists():
                drive.mount(str(mountpoint), force_remount=True)
        else:
            raise
    except Exception as exc:
        print(f'Colab Drive mount skipped or failed: {exc}')


def _is_repo_dir(path: Path) -> bool:
    return (
        (path / 'src' / 'sport_pipeline' / '__init__.py').exists()
        and (path / 'configs').exists()
        and (path / 'notebooks').exists()
    )


def _resolve_repo_dir() -> Path:
    fixed = Path('/content/drive/MyDrive/codex/batting_codex_handoff')
    candidates: list[Path] = []
    env_root = os.environ.get('BATTING_CODE_ROOT')
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend(
        [
            fixed,
            Path.cwd(),
        ]
    )
    candidates.extend(Path.cwd().parents)

    checked: list[str] = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            if candidate != fixed:
                print('WARNING: fixed REPO_DIR ではない場所の repo を使います。')
                print('  fixed:', fixed)
                print('  using:', candidate)
                print('  次回は repo フォルダを fixed path に置くか、BATTING_CODE_ROOT を明示してください。')
            return candidate

    raise FileNotFoundError(
        'sport_pipeline package が見つかりません。Drive には notebook 単体ではなく repo フォルダごと置く必要があります。\n'
        '推奨配置: /content/drive/MyDrive/codex/batting_codex_handoff\n'
        '別の場所に置いた場合は、この notebook の最初の cell より前に次を実行してください。\n'
        '  %env BATTING_CODE_ROOT=/content/drive/MyDrive/batting_codex_handoff\n'
        '確認した候補:\n- ' + '\n- '.join(checked)
    )


_mount_drive_if_needed()

REPO_DIR = _resolve_repo_dir()
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ.get('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME

BASE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)

src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if importlib.util.find_spec('sport_pipeline') is None:
    raise ModuleNotFoundError(
        'sport_pipeline を import できません。repo 配置と src/sport_pipeline/__init__.py を確認してください。\n'
        f'REPO_DIR={REPO_DIR}\n'
        f'src_dir={src_dir}\n'
        f'src_dir_exists={src_dir.exists()}'
    )

print('REPO_DIR =', REPO_DIR)
print('BASE_DIR =', BASE_DIR)
print('CACHE_DIR =', CACHE_DIR)
print('RUN_PROFILE_PATH =', RUN_PROFILE_PATH)
print('src_dir =', src_dir)

from sport_pipeline.pipeline.run_profile import artifact_id, load_run_profile, resolve_statcast_date_range, run_id, stage_settings, threshold

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
START_DATE, END_DATE = resolve_statcast_date_range(RUN_PROFILE)
FULL_RUN_ID = run_id(RUN_PROFILE, 'full_run_id')
PREDICTION_RUN_ID = run_id(RUN_PROFILE, 'sequence_tcn_run_id')
STRUCTURED_SEQUENCE_FEATURE_ID = artifact_id(RUN_PROFILE, 'structured_sequence_feature_id')
EVENT_WITH_PRIOR_DATASET_ID = artifact_id(RUN_PROFILE, 'event_with_prior_dataset_id')

print('STATCAST_DATE_RANGE =', START_DATE, 'to', END_DATE)
print('FULL_RUN_ID =', FULL_RUN_ID)
print('PREDICTION_RUN_ID =', PREDICTION_RUN_ID)
print('STRUCTURED_SEQUENCE_FEATURE_ID =', STRUCTURED_SEQUENCE_FEATURE_ID)


In [ ]:
from sport_pipeline.artifact_check import check_artifacts
from sport_pipeline.runtime.device import summarize_runtime_device

print(summarize_runtime_device(prefer_gpu=True, require_gpu=False))
required = [
    'manifests/bbe_events_v1.parquet',
    f'features/{STRUCTURED_SEQUENCE_FEATURE_ID}/manifest.parquet',
    f'features/{STRUCTURED_SEQUENCE_FEATURE_ID}/frames.parquet',
]
optional = [f'datasets/{EVENT_WITH_PRIOR_DATASET_ID}/manifest.parquet']
print(check_artifacts(BASE_DIR, required))
print('optional prior artifact:')
print(check_artifacts(BASE_DIR, optional))


In [ ]:
from sport_pipeline.io import read_table
from sport_pipeline.models.sequence.train_tcn import summarize_tcn_inputs

SEQUENCE_MANIFEST_PATH = BASE_DIR / 'features' / STRUCTURED_SEQUENCE_FEATURE_ID / 'manifest.parquet'
SEQUENCE_FRAMES_PATH = BASE_DIR / 'features' / STRUCTURED_SEQUENCE_FEATURE_ID / 'frames.parquet'
EVENT_WITH_PRIOR_PATH = BASE_DIR / 'datasets' / EVENT_WITH_PRIOR_DATASET_ID / 'manifest.parquet'
summary = summarize_tcn_inputs(
    BASE_DIR,
    sequence_manifest=SEQUENCE_MANIFEST_PATH,
    frames=SEQUENCE_FRAMES_PATH,
    event_with_prior=EVENT_WITH_PRIOR_PATH,
)
print(summary)

if summary['row_counts']['matched_trainable_sequences'] == 0:
    raise RuntimeError(
        '18 cannot train because there are 0 matched structured sequence samples. '
        f"reason={summary['likely_stop_reason']}\n"
        'Check that 12 produced non-empty clips_v1, then run 17 to rebuild '
        'features/structured_sequence_v1 before training. '
        'If 17 was run while clips_v1 was empty, it may have overwritten the sequence feature artifacts with empty files.'
    )


In [ ]:
from sport_pipeline.models.sequence.train_tcn import run_tcn_training

TCN_SETTINGS = stage_settings(RUN_PROFILE, 'sequence_tcn')
MODEL_FAMILY = str(TCN_SETTINGS.get('model_family', 'tcn'))  # 'ms_tcn' is also supported
MAX_EPOCHS = int(TCN_SETTINGS.get('max_epochs', 20))
HIDDEN_DIM = int(TCN_SETTINGS.get('hidden_dim', 64))
DEPTH = int(TCN_SETTINGS.get('depth', 3))
DROPOUT = float(TCN_SETTINGS.get('dropout', 0.10))
CONTACT_AUX_WEIGHT = float(TCN_SETTINGS.get('contact_aux_weight', 0.20))
DEVICE = str(TCN_SETTINGS.get('device', 'auto'))
PRIOR_FEATURE_MODE = str(TCN_SETTINGS.get('prior_feature_mode', 'concat_if_available'))
RESUME_TRAINING = bool(TCN_SETTINGS.get('resume', True))
CHECKPOINT_EVERY_EPOCH = bool(TCN_SETTINGS.get('checkpoint_every_epoch', True))

outputs = run_tcn_training(
    BASE_DIR,
    prediction_run_id=PREDICTION_RUN_ID,
    sequence_manifest=SEQUENCE_MANIFEST_PATH,
    frames=SEQUENCE_FRAMES_PATH,
    event_with_prior=EVENT_WITH_PRIOR_PATH,
    model_family=MODEL_FAMILY,
    hidden_dim=HIDDEN_DIM,
    depth=DEPTH,
    max_epochs=MAX_EPOCHS,
    dropout=DROPOUT,
    contact_aux_weight=CONTACT_AUX_WEIGHT,
    device=DEVICE,
    prior_feature_mode=PRIOR_FEATURE_MODE,
    resume=RESUME_TRAINING,
    checkpoint_every_epoch=CHECKPOINT_EVERY_EPOCH,
)
for name, path in outputs.items():
    print(name, '->', path, 'exists=', path.exists())
print('TCN progress is saved during training at:', outputs['progress'])
print('TCN checkpoint is saved during training at:', outputs['checkpoint'])


In [ ]:
expected = [
    f'predictions/{PREDICTION_RUN_ID}/predictions_v1.parquet',
    f'predictions/{PREDICTION_RUN_ID}/metrics_v1.json',
    f'reports/preflight/train_tcn_{PREDICTION_RUN_ID}.json',
    f'reports/preflight/train_tcn_{PREDICTION_RUN_ID}_progress.json',
    f'models/sequence/{PREDICTION_RUN_ID}/checkpoint.pt',
]
print(check_artifacts(BASE_DIR, expected))
print('NEXT: notebooks/19_frozen_visual_encoder.ipynb or notebooks/15_full_fusion.ipynb')
